# 【Day 08 實作】一句話 Prompt 和 CO-STAR，寫出來會差多少？

## Goal

[Day 8 文章](./day08.md) 提到：Prompt Engineering 的重點不是尋找神奇咒語，而是把工作需要的情境、目標、限制與交付形式說清楚。

這份 notebook 使用同一個 Groq 模型 `openai/gpt-oss-120b`，比較兩個 Prompt：

1. 一句模糊要求：`幫我寫一篇介紹生成式 AI 的文章。`
2. Day 8 文章中的完整 **CO-STAR** Prompt。

兩份文章都會完整輸出並渲染成 Markdown。請先閱讀文章，再看結構檢查與人工觀察問題。

> **實驗定位：**這是單一模型、兩個 Prompt、各執行一次的探索性教學示範。它可以呈現「補充資訊後，這次輸出如何改變」，但不能證明 CO-STAR 或任何 Prompt 框架必然比較好。


## Setup

請從 repo 根目錄或 `Day8` 目錄啟動 notebook，並把 Groq API Key 放在專案根目錄的 `.env`：

```text
GROQ_API_KEY=你的金鑰
```

這份 notebook 會呼叫 Groq 兩次。不要把 API Key 直接寫進程式碼、輸出或 Git。

目前使用的模型 ID 是 `openai/gpt-oss-120b`。模型權限與 Rate Limits 可能依帳戶或專案設定不同，請以自己的 Groq Console 為準。


In [1]:
%pip install -q groq python-dotenv


Note: you may need to restart the kernel to use updated packages.



In [2]:
import os
import re
from importlib.metadata import PackageNotFoundError, version

import groq
from dotenv import find_dotenv, load_dotenv
from groq import Groq
from IPython.display import Markdown, display

load_dotenv(find_dotenv(usecwd=True))

MODEL_ID = "openai/gpt-oss-120b"
REASONING_EFFORT = "low"
MAX_COMPLETION_TOKENS = 4500


def package_version(distribution_name: str) -> str:
    try:
        return version(distribution_name)
    except PackageNotFoundError:
        return "not installed"


print(f"模型：{MODEL_ID}")
print(f"Reasoning effort：{REASONING_EFFORT}")
print(f"groq {package_version('groq')} / python-dotenv {package_version('python-dotenv')}")


模型：openai/gpt-oss-120b
Reasoning effort：low
groq 1.6.0 / python-dotenv 1.1.0


## Step 1 — 先說清楚怎麼比較

為了讓差異主要來自 Prompt，兩次請求固定使用：

- 相同模型：`openai/gpt-oss-120b`
- 相同 `reasoning_effort="low"`
- 相同最大輸出長度
- 各自開一個新的單輪對話，不沿用前一次回答

但這仍不是嚴謹的因果實驗：

- CO-STAR Prompt 不只「格式不同」，還提供了更多背景與限制。
- 生成模型即使收到相同 Prompt，也可能產生不同答案。
- 每組只執行一次，不能估計穩定性或平均表現。

因此，請把它當成「看見需求資訊如何影響一次輸出」的示範，而不是框架排行榜。


## Step 2 — 準備兩個 Prompt

### A. 一句話 Prompt

只有任務，沒有交代讀者、用途、語氣、長度或文章結構。

### B. CO-STAR Prompt

直接使用 [day08.md](./day08.md) 中的範例，完整指定 Context、Objective、Style、Tone、Audience 與 Response。


In [3]:
BASIC_PROMPT = "幫我寫一篇介紹生成式 AI 的文章。"

COSTAR_PROMPT = """## Context

我要替大學通識課程撰寫一篇生成式 AI 入門文章。
讀者大多使用過 ChatGPT，但沒有程式設計或機器學習背景。

## Objective

用生活化方式解釋生成式 AI 如何根據使用者輸入產生文字，
並說明它適合與不適合處理的任務。

## Style

採用科普文章的寫法。
先用生活類比建立理解，再補上必要的正式名詞。

## Tone

親切、清楚，可以加入少量幽默，但不要使用浮誇的宣傳語氣。

## Audience

18 至 22 歲的大學生，對 AI 有興趣，但尚未系統性學習相關知識。

## Response

使用 Markdown，控制在 1,200 字以內，包含：

1. 一個生活化的開場
2. 生成式 AI 的基本原理
3. 三個適合使用的情境
4. 三個需要小心的限制
5. 一個讓讀者反思的結尾問題
"""

display(Markdown(f"### A. 一句話 Prompt\n\n```text\n{BASIC_PROMPT}\n```"))
display(Markdown(f"### B. CO-STAR Prompt\n\n```markdown\n{COSTAR_PROMPT}\n```"))


### A. 一句話 Prompt

```text
幫我寫一篇介紹生成式 AI 的文章。
```

### B. CO-STAR Prompt

```markdown
## Context

我要替大學通識課程撰寫一篇生成式 AI 入門文章。
讀者大多使用過 ChatGPT，但沒有程式設計或機器學習背景。

## Objective

用生活化方式解釋生成式 AI 如何根據使用者輸入產生文字，
並說明它適合與不適合處理的任務。

## Style

採用科普文章的寫法。
先用生活類比建立理解，再補上必要的正式名詞。

## Tone

親切、清楚，可以加入少量幽默，但不要使用浮誇的宣傳語氣。

## Audience

18 至 22 歲的大學生，對 AI 有興趣，但尚未系統性學習相關知識。

## Response

使用 Markdown，控制在 1,200 字以內，包含：

1. 一個生活化的開場
2. 生成式 AI 的基本原理
3. 三個適合使用的情境
4. 三個需要小心的限制
5. 一個讓讀者反思的結尾問題

```

## Step 3 — 定義相同的模型呼叫方式

下方函式不會把推理過程顯示出來，只取最終文章。錯誤訊息會區分金鑰、限制、權限與連線問題，但不會印出 API Key。


In [4]:
def generate_article(prompt: str) -> tuple[str, dict]:
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError(
            "找不到 GROQ_API_KEY。請在專案根目錄的 .env 設定金鑰後重新執行。"
        )

    client = Groq(api_key=api_key)
    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            reasoning_effort=REASONING_EFFORT,
            reasoning_format="hidden",
            max_completion_tokens=MAX_COMPLETION_TOKENS,
            messages=[{"role": "user", "content": prompt}],
        )
    except groq.RateLimitError:
        raise RuntimeError(
            "Groq 已達目前帳戶的 Rate Limit；請查看 Console Limits 後稍後重試。"
        ) from None
    except groq.AuthenticationError:
        raise RuntimeError("GROQ_API_KEY 無效，請確認 .env 設定。") from None
    except groq.PermissionDeniedError:
        raise RuntimeError(
            f"目前帳戶或專案沒有使用 {MODEL_ID} 的權限，請檢查 Model Permissions。"
        ) from None
    except (groq.APIConnectionError, groq.APITimeoutError):
        raise RuntimeError("無法連線至 Groq，請確認網路後重試。") from None
    except groq.APIStatusError as error:
        raise RuntimeError(
            f"Groq API 回傳錯誤：{error.status_code} {error.message}"
        ) from None

    choices = getattr(response, "choices", None)
    if not choices:
        raise RuntimeError("Groq 回應缺少 choices，無法取得文章。")

    choice = choices[0]
    content = getattr(getattr(choice, "message", None), "content", None)
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError("Groq 沒有回傳可用的文章內容。")
    if getattr(choice, "finish_reason", None) == "length":
        raise RuntimeError(
            "文章被 max_completion_tokens 截斷，不能拿來做這次比較。"
        )

    usage = getattr(response, "usage", None)
    usage_summary = {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }
    return content.strip(), usage_summary


def render_article(title: str, article: str) -> None:
    display(Markdown(f"### {title}\n\n---\n\n{article}\n\n---"))


## Step 4 — 一般 Prompt 的完整輸出

先完整閱讀模型這次如何自行決定受眾、深度、語氣與版面。不要急著只看字數或標題數。


In [5]:
basic_article, basic_usage = generate_article(BASIC_PROMPT)
render_article("A. 一句話 Prompt 產生的文章", basic_article)
print(f"Token usage：{basic_usage}")


### A. 一句話 Prompt 產生的文章

---

## 生成式 AI（Generative AI）概述  
*作者：ChatGPT（2026 年 8 月）*  

### 1. 什麼是生成式 AI？

生成式 AI（Generative Artificial Intelligence）是一類 **能夠自行創造新內容** 的機器學習模型。與傳統的辨識、分類或回歸模型不同，生成式 AI 的目標不是「判斷」而是「產生」。它可以根據訓練資料的分布，生成文字、圖像、音頻、程式碼、甚至 3D 模型等多媒體形式的作品。

常見的生成式 AI 框架包括：

| 類型 | 代表模型 | 主要技術 | 典型應用 |
|------|----------|----------|----------|
| 文本生成 | GPT‑4、Claude、LLaMA、Mistral | 大規模語言模型（Transformer） | 文章寫作、客服機器人、程式碼補全 |
| 圖像生成 | DALL·E、Stable Diffusion、Midjourney | Diffusion Model、GAN | 插畫、概念設計、廣告素材 |
| 音頻/語音生成 | AudioLM、ChatTTS、RVC | WaveNet、Diffusion、Flow-based | 語音助理、配音、音樂創作 |
| 多模態生成 | Flamingo、Gemini、GPT‑4V | 多模態 Transformer + Vision-Language 交叉注意力 | 圖文互動、視訊腳本生成 |
| 3D/動畫生成 | DreamFusion、Shap-E、Meta’s Make‑It‑3D | Neural Radiance Fields (NeRF) + Diffusion | 虛擬場景、遊戲資產、AR/VR 內容 |

### 2. 生成式 AI 的核心技術

| 技術 | 原理簡述 | 為何適合生成 |
|------|----------|---------------|
| **Transformer** | 透過自注意力機制同時關注序列中所有位置的資訊，能捕捉長距離依賴。 | 在文字、程式碼、甚至圖像（Vision Transformer）上表現出色，適合大規模語言與視覺生成。 |
| **擴散模型（Diffusion Model）** | 從噪聲逐步「逆向擴散」回復目標資料分布，訓練時學習噪聲去除的過程。 | 生成高品質、細節豐富的圖像、音頻，且訓練相對穩定。 |
| **生成對抗網路（GAN）** | 由生成器與判別器對抗學習，生成器試圖騙過判別器。 | 雖然訓練不易，但在生成逼真圖像、影片方面仍有競爭力。 |
| **自回歸模型（Auto‑Regressive）** | 逐步預測序列的下一个 token，條件於先前已生成的內容。 | 文字、音頻等序列資料的自然生成方式。 |
| **混合模型（Hybrid）** | 把 Transformer、Diffusion、Flow 等技術結合，互補優勢。 | 例如：在文本引導下的圖像擴散（text‑to‑image）或在音頻上加入語意控制。 |

### 3. 生成式 AI 的典型應用場景

| 領域 | 具體應用 | 商業價值 |
|------|----------|----------|
| **內容創作** | 文章、博客、廣告文案、小說、劇本 | 大幅降低文字創作成本、加速內容迭代。 |
| **設計與藝術** | 圖像、插畫、品牌 LOGO、3D 產品概念 | 讓非設計師也能快速產出視覺稿，縮短概念驗證時間。 |
| **程式開發** | 程式碼補全、單元測試生成、錯誤診斷 | 提高開發效率、減少人為錯誤。 |
| **客服與助理** | 智能聊天機器人、知識庫自動摘要 | 24/7 服務、降低客服人力成本。 |
| **教育與培訓** | 個性化學習材料、即時解題指導 | 提升學習體驗、減輕教師負擔。 |
| **醫療** | 病歷摘要、藥物分子結構生成、影像診斷輔助 | 加速新藥研發、提升診斷效率。 |
| **娛樂** | 音樂、虛擬人物對話、遊戲關卡自動生成 | 擴充內容產量、提升玩家沉浸感。 |
| **金融** | 報告撰寫、風險情景模擬、合規文檔自動生成 | 減少人工審核時間、提高合規性。 |

### 4. 生成式 AI 的挑戰與風險

| 風險類別 | 具體問題 | 可能的緩解措施 |
|----------|----------|----------------|
| **倫理與濫用** | 假新聞、深偽（Deepfake）影像、版權侵害 | 建立內容驗證水印、法規制定、AI 使用者教育。 |
| **偏見與公平** | 訓練資料中的種族、性別、文化偏見會在生成內容中顯現。 | 多樣化資料收集、偏見檢測工具、後處理過濾。 |
| **可解釋性** | 生成結果難以追溯來源或原因。 | 研發可視化注意力圖、反事實檢測（counterfactual analysis）。 |
| **資源消耗** | 大模型訓練與推理需要大量 GPU/TPU 計算，能耗高。 | 模型蒸餾、稀疏化、邊緣推理（Edge AI）技術。 |
| **法律責任** | 生成內容侵權或誤導，誰該負責？ | 明確 AI 產出歸屬規範、使用者合約中加入 AI 免責條款。 |
| **安全性** | Prompt injection、模型竊取或惡意微調。 | 沙箱執行、模型加密、使用者身份驗證與審計。 |

### 5. 產業趨勢與未來展望

1. **模型即服務（Model‑as‑a‑Service, MaaS）**  
   - 以 API 形式提供的生成模型（OpenAI、Anthropic、Google Gemini）已成為企業快速接入的主要途徑。未來會出現更多 **垂直領域專用** 的模型服務（法律、醫療、金融等）。

2. **多模態融合**  
   - 從單純文字或圖像生成，發展到 **文字‑圖像‑音頻‑3D** 全方位創作平台。跨模態的「敘事生成」將支撐虛擬製作（Metaverse）與沉浸式教育。

3. **個性化生成**  
   - 通過少量樣本微調（LoRA、Adapter）或檔案檢索增強（RAG），實現 **針對個人或企業風格的定制化生成**。這將是 AI 內容差異化競爭的關鍵。

4. **可持續與綠色 AI**  
   - 隨著 ESG（環境、社會、治理）要求升高，**低功耗模型、能效指標（ECO‑Score）** 會成為選型重要參考。

5. **法規與標準化**  
   - 歐盟 AI 法、ISO/IEC 42001（AI 風險管理）等國際標準正在形成。企業需要在 **合規、審計、風險治理** 上做好前置布局。

### 6. 入門指南：如何自己玩轉生成式 AI

| 步驟 | 操作說明 | 推薦資源 |
|------|----------|----------|
| 1️⃣ 確定目標 | 文字、圖像、音頻或多模態？ | 《Generative AI Playbook》（2024） |
| 2️⃣ 選擇模型 | 開源：Stable Diffusion、LLaMA、AudioLM；雲端：OpenAI API、Google Gemini | Hugging Face Hub、Model Zoo |
| 3️⃣ 準備環境 | Python 3.10+、CUDA 驅動、`torch`、`diffusers`、`transformers` | 官方 Docker 镜像、Colab Notebook |
| 4️⃣ 資料收集與清洗 | 若要微調，需要高品質、標註完整的資料集 | LAION‑5B（圖像）、The Pile（文字） |
| 5️⃣ 微調或提示工程 | LoRA、QLoRA、Prompt‑Engineering | 《Effective Prompting》（2023） |
| 6️⃣ 部署與測試 | FastAPI + Uvicorn → 部署為 REST API；或使用 Gradio 搭建 UI | FastAPI docs、Gradio Gallery |
| 7️⃣ 監控與治理 | 記錄 Prompt、輸出、用戶反饋；設定安全過濾 | LangChain + LLMGuard、OpenAI Moderation API |

**小範例：用 Python 生成一張「未來城市夜景」圖像**

```python
# 安裝依賴
# pip install diffusers transformers torch accelerate

import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1",
    torch_dtype=torch.float16,
    revision="fp16"
).to("cuda")

prompt = "a futuristic cyber‑punk city at night, neon lights, ultra‑realistic, high detail"
image = pipe(prompt, num_inference_steps=50, guidance_scale=7.5).images[0]

image.save("future_city.png")
print("圖像已儲存為 future_city.png")
```

> **提示技巧**：加入風格詞（“cinematic”, “ultra‑realistic”）和光線描述（“soft rim lighting”）能顯著提升畫質。

### 7. 結語

生成式 AI 正從 **「研究原型」** 迅速演化為 **「日常工具」**。它不僅改變內容創作的方式，也在重塑產品設計、程式開發、教育培訓等整個產業鏈。面對技術紅利，我們同時必須正視倫理、偏見與資源消耗等挑戰，透過合規、技術治理與跨領域合作，才能讓生成式 AI 為社會帶來持續、可負擔且安全的價值。

> **一句話總結**：生成式 AI 讓「創造」不再是少數專家的專利，而是每個人都可以即時、低成本完成的日常能力——只要我們懂得正確使用、負責任管理。  

---  

*本文旨在提供生成式 AI 的全景概覽，若需更深入的技術實作或產業案例，歡迎隨時提出具體需求。*

---

Token usage：{'prompt_tokens': 84, 'completion_tokens': 2635, 'total_tokens': 2719}


## Step 5 — CO-STAR Prompt 的完整輸出

接著閱讀同一模型收到完整工作說明後的文章。留意它是否真的遵守條件，而不只是版面看起來更整齊。


In [6]:
costar_article, costar_usage = generate_article(COSTAR_PROMPT)
render_article("B. CO-STAR Prompt 產生的文章", costar_article)
print(f"Token usage：{costar_usage}")


### B. CO-STAR Prompt 產生的文章

---

## 1️⃣ 生活化的開場：AI 像「會說話的自助餐廚師」  

想像你走進自助餐廳，面前有一位廚師，他會根據你說的「我想要鹹、帶點酸、又不太辣」馬上在爐子前揮灑，端出一道看起來像是你專屬的料理。  
你不需要告訴他食譜的每一步，只要給出大概的需求，廚師就會把過去學過的料理技巧、調味料比例、食材搭配全都拿出來，快速「即興」完成。  

生成式 AI（Generative AI）在文字領域的工作方式，就是類似這位「會說話的自助餐廚師」：**根據使用者的提示（prompt），即時組合、重組大量已學過的文字資料，產出看起來像是人類寫出的回應**。接下來，我們用更具體的概念說明它是怎麼運作的，並探討「什麼情境適合」與「什麼情境要小心」。

---

## 2️⃣ 生成式 AI 的基本原理（不拋硬核公式）

| 名詞 | 簡易說明 | 為什麼重要 |
|------|----------|------------|
| **語料庫（corpus）** | AI 先在網路、書本、論文等大量文字資料上「閱讀」和「記憶」——就像廚師在料理學校裡吃遍世界料理的教材。 | 決定模型能產生什麼樣的知識與語言風格。 |
| **模型（model）** | 一組由數十億個「參數」組成的神經網路，負責把文字轉換成數字、再把數字轉回文字。 | 參數越多，模型在捕捉語言細節上越靈活。 |
| **預訓練（pre‑training）** | 先讓模型在海量文字上自行預測下一個詞，學會語法、常識與常見的說話方式。 | 相當於廚師在學校練習「隨手炒」的基礎功。 |
| **微調（fine‑tuning）** | 再把模型針對特定領域（法律、醫學、程式）做進一步訓練，讓它更懂專業用語。 | 如同廚師在某家餐廳專門練習該店的招牌菜。 |
| **提示（prompt）** | 使用者給模型的文字指令。好比你對廚師說「想要酸甜口的義大利麵」。 | 提示的寫法會直接影響最終「料理」的味道與品質。 |
| **解碼策略（sampling）** | 決定模型在每一步選哪個字／詞，常見有「貪婪取最大機率」或「隨機抽樣」等。 | 控制回應的創意度與一致性。 |

**一句話概括**：生成式 AI 先在大量文字中學會「怎麼寫」，再根據你的提示即時「下廚」產出答案。  

---

## 3️⃣ 三個適合使用生成式 AI 的情境  

| 情境 | 為什麼合適 | 小技巧 |
|------|------------|--------|
| **① 文字草稿與靈感激盪**<br>（寫作、簡報、社群貼文） | AI 能快速提供多種說法、標題或段落結構，幫助你突破「腦袋卡住」的瓶頸。 | 把 AI 產出的文字當作 **草稿**，再自行編輯、加入個人風格。 |
| **② 語言學習與翻譯練習**<br>（單字解釋、文法示例） | 只要輸入「請用簡單中文說明『quantum entanglement』」即可得到易懂解說，或請 AI 產生雙語例句。 | 設定 **角色**（如「扮演英語老師」）讓回應更具教學性。 |
| **③ 快速原型與程式碼範例**<br>（簡單腳本、SQL 查詢） | 即使不會寫程式，輸入「把 Excel 表格匯入 MySQL」的需求，AI 能產出基本程式碼，讓你先驗證概念。 | 產出後一定 **自行測試**，不要直接在正式環境執行。 |

---

## 4️⃣ 三個需要小心的限制  

| 限制 | 為什麼要注意 | 實務建議 |
|------|---------------|----------|
| **① 事實錯誤（Hallucination）** | AI 只能根據訓練資料「猜」答案，可能會編造不存在的統計數字或引用虛構的文獻。 | **核對**：對重要資訊（數據、引用、法律條文）一定要自行查證。 |
| **② 版權與偏見** | 訓練資料中可能包含受版權保護的內容或帶有社會偏見的語句，AI 產出時會不自覺地複製這些特徵。 | 避免將 AI 產出的長篇文字直接當作作業或出版物，必要時 **重寫、引用來源**。 |
| **③ 缺乏情境深度** | AI 不了解「情感、道德、法律」的真正含義，只會模仿文字形式。對於諮詢醫療、法律或心理問題，答案往往不具備專業可靠性。 | 把 AI 當作 **輔助工具**，重要決策前務必諮詢專業人士。 |

---

## 5️⃣ 結尾的反思問題  

> **如果 AI 能幫你寫好所有報告與演講稿，你會選擇把時間花在什麼其他事情上？**  

思考：AI 把「文字產出」的負擔減輕了，我們是否能利用這段「省下來」的時間，去探索更深層的批判思考、實驗創作，或是人際互動？這不僅是技術的問題，更是我們如何重新安排「學習」與「生活」的挑戰。

---

Token usage：{'prompt_tokens': 312, 'completion_tokens': 1542, 'total_tokens': 1854}


## Step 6 — 可量化的結構檢查

下方只檢查容易量化的表面結構：

- 去除 Markdown 標記後的約略字元數
- Markdown 標題數
- 條列與編號項目數
- 結尾附近是否有問號
- API 回報的 Token usage

這些數字可以協助定位差異，但不能直接代表文章比較正確、自然或有用。


In [7]:
def approximate_visible_characters(text: str) -> int:
    without_code_fences = re.sub(r"```.*?```", "", text, flags=re.DOTALL)
    without_links = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", without_code_fences)
    without_markers = re.sub(r"[#*_>`~-]", "", without_links)
    return len(re.sub(r"\s+", "", without_markers))


def article_metrics(article: str, usage: dict) -> dict:
    lines = article.splitlines()
    ending = article[-200:]
    return {
        "約略可見字元": approximate_visible_characters(article),
        "Markdown 標題數": sum(
            bool(re.match(r"^#{1,6}\s+", line)) for line in lines
        ),
        "條列項目數": sum(
            bool(re.match(r"^\s*[-*+]\s+", line)) for line in lines
        ),
        "編號項目數": sum(
            bool(re.match(r"^\s*\d+[.)]\s+", line)) for line in lines
        ),
        "結尾 200 字內有問號": "是" if ("？" in ending or "?" in ending) else "否",
        "Prompt tokens": usage.get("prompt_tokens"),
        "Completion tokens": usage.get("completion_tokens"),
    }


basic_metrics = article_metrics(basic_article, basic_usage)
costar_metrics = article_metrics(costar_article, costar_usage)

rows = "\n".join(
    f"| {metric} | {basic_metrics[metric]} | {costar_metrics[metric]} |"
    for metric in basic_metrics
)
display(
    Markdown(
        "### 本次結構比較\n\n"
        "| 項目 | 一句話 Prompt | CO-STAR Prompt |\n"
        "| --- | ---: | ---: |\n"
        f"{rows}\n\n"
        "> 約略可見字元只用來比較篇幅，不等同精確中文『字數』。"
    )
)


### 本次結構比較

| 項目 | 一句話 Prompt | CO-STAR Prompt |
| --- | ---: | ---: |
| 約略可見字元 | 3088 | 1600 |
| Markdown 標題數 | 10 | 5 |
| 條列項目數 | 5 | 0 |
| 編號項目數 | 5 | 0 |
| 結尾 200 字內有問號 | 否 | 是 |
| Prompt tokens | 84 | 312 |
| Completion tokens | 2635 | 1542 |

> 約略可見字元只用來比較篇幅，不等同精確中文『字數』。

## Step 7 — 人工觀察清單

請回到兩篇完整文章，逐項觀察：

| 問題 | 一句話 Prompt | CO-STAR Prompt |
| --- | --- | --- |
| 能否看出文章寫給誰？ | 自行記錄 | 自行記錄 |
| 是否先用生活類比建立理解？ | 自行記錄 | 自行記錄 |
| 是否清楚解釋必要名詞？ | 自行記錄 | 自行記錄 |
| 是否真的列出三個適用情境？ | 自行記錄 | 自行記錄 |
| 是否真的列出三個限制？ | 自行記錄 | 自行記錄 |
| 語氣是否親切但不浮誇？ | 自行記錄 | 自行記錄 |
| 結尾是否提出值得思考的問題？ | 自行記錄 | 自行記錄 |
| 是否一致使用繁體中文？ | 自行記錄 | 自行記錄 |
| 是否遵守 1,200 字限制？ | 未指定 | 自行記錄 |
| 哪些事實主張需要查核？ | 自行記錄 | 自行記錄 |

這些判斷需要閱讀內容，不能只靠關鍵字或同一個模型替自己打分。


## Step 8 — 如何解讀這次結果

本次可以觀察：

- 模型在資訊較少時，替使用者自行決定了哪些條件。
- 提供受眾、風格、語氣與格式後，哪些內容或結構發生改變。
- CO-STAR Prompt 中有哪些條件被遵守、忽略或只做到表面。

本次不能證明：

- CO-STAR 必然比短 Prompt 好。
- 文章比較長、標題比較多，就代表品質比較高。
- GPT-OSS 120B 每次都會產生相同差異。
- 所有任務都需要完整填寫六個 CO-STAR 欄位。

真正的重點不是背下六個英文字母，而是辨認：這項工作還缺少哪些會影響成果的資訊？


## Next Steps

可以延伸做三個小實驗：

1. 對兩個 Prompt 各執行 3 次，觀察輸出的穩定性。
2. 只增加 Audience，看看一個欄位能改變多少。
3. 保留 CO-STAR，但移除 Response，觀察版面與交付形式如何改變。

### 延伸資料

- [Day 8 文章](./day08.md)
- [Groq：GPT-OSS 120B](https://console.groq.com/docs/model/openai/gpt-oss-120b)
- [Groq：Reasoning](https://console.groq.com/docs/reasoning)
- [Groq：Rate Limits](https://console.groq.com/docs/rate-limits)
- [CO-STAR 原始介紹文章](https://medium.com/data-science/how-i-won-singapores-gpt-4-prompt-engineering-competition-34c195a93d41)
